# Config

## Add project root to Python path inside the notebook

In [1]:
import sys, os

# Go one level up from the notebook folder to project root
project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project root added:", project_root)

Project root added: c:\Users\atulsehgal\OneDrive\Documents\repos\talk-to-my-data-semantic


In [2]:
from utils.config_loader import load_env
load_env()

✅ Loaded environment variables from configs/dev.env


# Validation

## Test- Semantic model loads correctly

In [3]:
from src.semantic.semantic_model import SemanticModel

model = SemanticModel.from_yaml(
    os.path.join(project_root, "src/semantic/model_tpch.yml")
)

print(model.tables.keys())
print(model.measures.keys())
print(model.dimensions.keys())
print(model.relationships[0])

dict_keys(['customer', 'orders', 'lineitem', 'part', 'partsupp', 'supplier', 'nation', 'region'])
dict_keys(['revenue', 'gross_sales', 'order_count', 'avg_discount', 'quantity_sold', 'average_order_size', 'average_quantity', 'average_revenue', 'profit', 'profit_margin_pct', 'discount_amount', 'average_order_value', 'customer_count', 'supplier_count', 'arpc', 'line_count', 'average_lines_per_order'])
dict_keys(['order_date', 'ship_date', 'customer_name', 'customer_segment', 'region_name', 'nation_name', 'supplier_name', 'part_name', 'product_brand'])
Relationship(from_table='orders', from_column='o_custkey', to_table='customer', to_column='c_custkey', type='many_to_one', role='customer_hierarchy', description='Each order belongs to a single customer.')


## Test- Semantic resolver

In [4]:
from src.semantic.semantic_resolver import SemanticResolver
resolver = SemanticResolver(model)

In [5]:
from dataclasses import asdict
import json

Evaluate each one for:

✔ Correct measure selection

✔ Correct dimension selection

✔ Correct time filter

✔ Correct grouping

✔ Correct semantic matching

✔ Whether the LLM respected your model’s rules

✔ Whether join-related fields were interpreted correctly

✔ Any inconsistencies or potential improvements

=============== EXAMPLE 1 =================

In [6]:
plan = resolver.plan_from_question("sales in last 3 months")

print(json.dumps(asdict(plan), indent=2))

{
  "measure": {
    "name": "gross_sales",
    "expression": "SUM(l_extendedprice)",
    "table": "lineitem",
    "description": "Gross sales before discount.",
    "synonyms": [
      "gross_revenue"
    ]
  },
  "time_dimension": {
    "name": "order_date",
    "table": "orders",
    "column": "o_orderdate",
    "description": "Order entry date.",
    "synonyms": [
      "date",
      "transaction_date",
      "order_day"
    ],
    "time_grains": [
      "day",
      "month",
      "quarter",
      "year"
    ]
  },
  "time_filter": "order_date >= DATEADD(MONTH, -3, CURRENT_DATE)",
  "group_by_dimensions": [],
  "time_grain": null
}


⭐ Final Verdict

✔ Correct measure

✔ Correct time dimension

✔ Correct time filter

✔ Correct grouping (none)

✔ No errors or hallucinations

❗ SQL column names will be corrected in Step 5

This is a solid semantic reasoning output.

=============== EXAMPLE 2 =================

In [7]:
plan = resolver.plan_from_question("top 5 customers by revenue")

print(json.dumps(asdict(plan), indent=2))

{
  "measure": {
    "name": "revenue",
    "expression": "SUM(l_extendedprice * (1 - l_discount))",
    "table": "lineitem",
    "description": "Net sales revenue after discount at line level.",
    "synonyms": [
      "sales",
      "net_sales",
      "turnover",
      "sales_revenue"
    ]
  },
  "time_dimension": null,
  "time_filter": null,
  "group_by_dimensions": [
    {
      "name": "customer_name",
      "table": "customer",
      "column": "c_name",
      "description": "Customer name.",
      "synonyms": [
        "customer",
        "client",
        "buyer",
        "purchaser"
      ],
      "time_grains": []
    }
  ],
  "time_grain": null
}


⭐ Final Verdict

**This output is 95% perfect.

VERY GOOD semantic reasoning.**

✔ Correct measure

✔ Correct grouping

✔ No unnecessary time dimension

✔ No unnecessary time filter

❗ Minor improvement needed for LIMIT 5 (handled later)

=============== EXAMPLE 3 =================

In [8]:
plan = resolver.plan_from_question("revenue by month this year")

print(json.dumps(asdict(plan), indent=2))

{
  "measure": {
    "name": "revenue",
    "expression": "SUM(l_extendedprice * (1 - l_discount))",
    "table": "lineitem",
    "description": "Net sales revenue after discount at line level.",
    "synonyms": [
      "sales",
      "net_sales",
      "turnover",
      "sales_revenue"
    ]
  },
  "time_dimension": {
    "name": "order_date",
    "table": "orders",
    "column": "o_orderdate",
    "description": "Order entry date.",
    "synonyms": [
      "date",
      "transaction_date",
      "order_day"
    ],
    "time_grains": [
      "day",
      "month",
      "quarter",
      "year"
    ]
  },
  "time_filter": "order_date >= DATE_TRUNC('year', CURRENT_DATE)",
  "group_by_dimensions": [],
  "time_grain": "month"
}


⭐ Final Verdict

✔ Correct measure

✔ Correct time dimension

✔ Correct MTD time filter (perfect SQL logic)

✔ Correct grouping

✔ No hallucinations

✔ No unnecessary joins or dims

❗ Column name translation deferred to SQL Generator

❗ Grain not explicit yet (will be added later)

This is a very strong semantic interpretation (95%+ correct).

=============== EXAMPLE 4 =================

In [9]:
plan = resolver.plan_from_question("units sold by brand last quarter")

print(json.dumps(asdict(plan), indent=2))

{
  "measure": {
    "name": "quantity_sold",
    "expression": "SUM(l_quantity)",
    "table": "lineitem",
    "description": "Total quantity sold.",
    "synonyms": [
      "volume",
      "units_sold"
    ]
  },
  "time_dimension": {
    "name": "order_date",
    "table": "orders",
    "column": "o_orderdate",
    "description": "Order entry date.",
    "synonyms": [
      "date",
      "transaction_date",
      "order_day"
    ],
    "time_grains": [
      "day",
      "month",
      "quarter",
      "year"
    ]
  },
  "time_filter": "order_date >= DATE_TRUNC('quarter', CURRENT_DATE) - INTERVAL '3 months' AND order_date < DATE_TRUNC('quarter', CURRENT_DATE)",
  "group_by_dimensions": [
    {
      "name": "product_brand",
      "table": "part",
      "column": "p_brand",
      "description": "Product brand",
      "synonyms": [
        "brand",
        "product_brand",
        "brand_name"
      ],
      "time_grains": []
    }
  ],
  "time_grain": null
}


⭐ Final Verdict

✔ Correct measure

✔ Correct time dimension

✔ Correct quarterly time filter

✔ Correct grouping (brand)

✔ Correct join inference

✔ Perfect synonym recognition

✔ No hallucinations

❗ Minor: SQL column-name translation will be handled in SQL generator

=============== EXAMPLE 5 =================

In [10]:
plan = resolver.plan_from_question("number of orders placed in the last 10 days")

print(json.dumps(asdict(plan), indent=2))

{
  "measure": {
    "name": "order_count",
    "expression": "COUNT(DISTINCT o_orderkey)",
    "table": "orders",
    "description": "Number of distinct orders.",
    "synonyms": [
      "transactions",
      "num_orders",
      "order_volume"
    ]
  },
  "time_dimension": {
    "name": "order_date",
    "table": "orders",
    "column": "o_orderdate",
    "description": "Order entry date.",
    "synonyms": [
      "date",
      "transaction_date",
      "order_day"
    ],
    "time_grains": [
      "day",
      "month",
      "quarter",
      "year"
    ]
  },
  "time_filter": "order_date >= CURRENT_DATE - INTERVAL '10 days'",
  "group_by_dimensions": [],
  "time_grain": null
}


⭐ Overall Verdict (same structured style)

✔ Correct measure

✔ Correct time dimension

✔ Correct 10-day rolling filter

✔ Correct grouping (none)

✔ Correct table-level interpretation

✔ No hallucinations

✔ No incorrect joins

❗ Column-name translation for SQL handled later

⭐ Overall: 100% correct

## Test- Join Graph

In [11]:
from src.semantic.join_graph import JoinGraph

In [12]:
jg = JoinGraph.from_model(model)

jg.print_graph()

orders:
   orders.o_custkey  →  customer.c_custkey
   orders.o_orderkey  →  lineitem.l_orderkey
   orders.o_orderkey  →  lineitem.l_orderkey

customer:
   customer.c_custkey  →  orders.o_custkey
   customer.c_nationkey  →  nation.n_nationkey

lineitem:
   lineitem.l_orderkey  →  orders.o_orderkey
   lineitem.l_orderkey  →  orders.o_orderkey
   lineitem.l_partkey  →  part.p_partkey
   lineitem.l_partkey  →  partsupp.ps_partkey
   lineitem.l_suppkey  →  partsupp.ps_suppkey

nation:
   nation.n_nationkey  →  customer.c_nationkey
   nation.n_nationkey  →  supplier.s_nationkey
   nation.n_regionkey  →  region.r_regionkey

supplier:
   supplier.s_nationkey  →  nation.n_nationkey
   supplier.s_suppkey  →  partsupp.ps_suppkey

region:
   region.r_regionkey  →  nation.n_regionkey

part:
   part.p_partkey  →  lineitem.l_partkey

partsupp:
   partsupp.ps_suppkey  →  supplier.s_suppkey
   partsupp.ps_partkey  →  lineitem.l_partkey
   partsupp.ps_suppkey  →  lineitem.l_suppkey



## Test- Relationship objects are loading role correctly

In [13]:
import importlib

import src.semantic.join_graph as join_graph_module
import src.semantic.join_resolver as join_resolver_module

importlib.reload(join_graph_module)
importlib.reload(join_resolver_module)

<module 'src.semantic.join_resolver' from 'c:\\Users\\atulsehgal\\OneDrive\\Documents\\repos\\talk-to-my-data-semantic\\src\\semantic\\join_resolver.py'>

### Confirm that your Relationship objects are loading role correctly

In [14]:
for rel in model.relationships:
    print(rel.from_table, rel.from_column, "→", rel.to_table, rel.to_column, "| role:", rel.role)

orders o_custkey → customer c_custkey | role: customer_hierarchy
lineitem l_orderkey → orders o_orderkey | role: order
orders o_orderkey → lineitem l_orderkey | role: order
customer c_nationkey → nation n_nationkey | role: customer_hierarchy
supplier s_nationkey → nation n_nationkey | role: supplier_hierarchy
nation n_regionkey → region r_regionkey | role: customer_hierarchy
lineitem l_partkey → part p_partkey | role: product_hierarchy
partsupp ps_suppkey → supplier s_suppkey | role: supplier_hierarchy
lineitem l_partkey → partsupp ps_partkey | role: product_hierarchy
lineitem l_suppkey → partsupp ps_suppkey | role: supplier_hierarchy


### Confirm that roles are being put into JoinEdges

In [15]:
for src, edges in jg.graph.items():
    for e in edges:
        if e.target == "region":
            print(e)

JoinEdge(source='nation', target='region', source_column='n_regionkey', target_column='r_regionkey', role='customer_hierarchy', description='Each nation is associated with a region.')


### Confirm that find_path() is receiving preferred roles

In [16]:
path = jg.find_path(
    start="lineitem",
    target="region",
    preferred_roles={"customer_hierarchy"}
)

for e in path:
    print(e.source, "→", e.target, "| role:", e.role)


lineitem → orders | role: order
orders → customer | role: customer_hierarchy
customer → nation | role: customer_hierarchy
nation → region | role: customer_hierarchy


## Test- Join Path Resolution

In [56]:
jg.show_path_raw("lineitem", "region")
jg.show_path_semantic("lineitem", "region", preferred_roles={"customer_hierarchy"})

jg.show_path_raw("orders", "supplier")
jg.show_path_semantic("orders", "supplier", preferred_roles={"supplier_hierarchy"})

jg.show_path_raw("lineitem", "part")
jg.show_path_semantic("lineitem", "part")

jg.show_path_raw("customer", "region")
jg.show_path_semantic("customer", "region", preferred_roles={"customer_hierarchy"})

jg.show_path_raw("lineitem", "partsupp")
jg.show_path_semantic("lineitem", "partsupp", preferred_roles={"supplier_hierarchy"})


RAW shortest path from 'lineitem' to 'region':
   lineitem.l_orderkey → orders.o_orderkey | role: order
   orders.o_custkey → customer.c_custkey | role: customer_hierarchy
   customer.c_nationkey → nation.n_nationkey | role: customer_hierarchy
   nation.n_regionkey → region.r_regionkey | role: customer_hierarchy

SEMANTIC path from 'lineitem' to 'region' (roles={'customer_hierarchy'}):
   lineitem.l_orderkey → orders.o_orderkey | role: order
   orders.o_custkey → customer.c_custkey | role: customer_hierarchy
   customer.c_nationkey → nation.n_nationkey | role: customer_hierarchy
   nation.n_regionkey → region.r_regionkey | role: customer_hierarchy

RAW shortest path from 'orders' to 'supplier':
   orders.o_custkey → customer.c_custkey | role: customer_hierarchy
   customer.c_nationkey → nation.n_nationkey | role: customer_hierarchy
   nation.n_nationkey → supplier.s_nationkey | role: supplier_hierarchy

SEMANTIC path from 'orders' to 'supplier' (roles={'supplier_hierarchy'}):
   order

Correct
This is the supplier geography path.

IMPORTANT NOTE:
There are two valid geography paths in TPCH:

Path A (via customer):
lineitem → orders → customer → nation → region

Path B (via supplier):
lineitem → supplier → nation → region


Both are correct in TPCH — depending on whether you mean:

customer’s region

supplier’s region

Right now, BFS is picking the shortest path, which is supplier region (3 hops) instead of customer region (4 hops).

➡️ This is correct per BFS definition.
➡️ Later, we will allow the resolver to choose the correct semantic path based on dimension context (e.g., region of customer vs region of supplier).

For now, your BFS is functioning perfectly.

In [18]:
jg.show_path_raw("orders", "supplier")
jg.show_path_raw("lineitem", "part")
jg.show_path_raw("customer", "region")


RAW shortest path from 'orders' to 'supplier':
   orders.o_custkey → customer.c_custkey | role: customer_hierarchy
   customer.c_nationkey → nation.n_nationkey | role: customer_hierarchy
   nation.n_nationkey → supplier.s_nationkey | role: supplier_hierarchy

RAW shortest path from 'lineitem' to 'part':
   lineitem.l_partkey → part.p_partkey | role: product_hierarchy

RAW shortest path from 'customer' to 'region':
   customer.c_nationkey → nation.n_nationkey | role: customer_hierarchy
   nation.n_regionkey → region.r_regionkey | role: customer_hierarchy


## Test- Build Join Path Resolver for SemanticPlan

In [19]:
from src.semantic.join_resolver import JoinResolver
from src.semantic.semantic_resolver import SemanticResolver

In [20]:
joiner = JoinResolver(jg)

In [21]:
plan = resolver.plan_from_question("revenue by region last year")
print(json.dumps(asdict(plan), indent=2))

{
  "measure": {
    "name": "revenue",
    "expression": "SUM(l_extendedprice * (1 - l_discount))",
    "table": "lineitem",
    "description": "Net sales revenue after discount at line level.",
    "synonyms": [
      "sales",
      "net_sales",
      "turnover",
      "sales_revenue"
    ]
  },
  "time_dimension": {
    "name": "order_date",
    "table": "orders",
    "column": "o_orderdate",
    "description": "Order entry date.",
    "synonyms": [
      "date",
      "transaction_date",
      "order_day"
    ],
    "time_grains": [
      "day",
      "month",
      "quarter",
      "year"
    ]
  },
  "time_filter": "order_date >= DATE_TRUNC('year', CURRENT_DATE) - INTERVAL '1 year' AND order_date < DATE_TRUNC('year', CURRENT_DATE)",
  "group_by_dimensions": [
    {
      "name": "region_name",
      "table": "region",
      "column": "r_name",
      "description": "Region name.",
      "synonyms": [
        "region",
        "geographic_region"
      ],
      "time_grains": []
  

In [22]:
edges = joiner.joins_for_plan(plan)

for edge in edges:
    print(edge)

print("\n")

for e in edges:
    print(f"{e.source}.{e.source_column} → {e.target}.{e.target_column}")

JoinEdge(source='lineitem', target='orders', source_column='l_orderkey', target_column='o_orderkey', role='order', description='Each line item belongs to a single order.')
JoinEdge(source='orders', target='customer', source_column='o_custkey', target_column='c_custkey', role='customer_hierarchy', description='Each order belongs to a single customer.')
JoinEdge(source='customer', target='nation', source_column='c_nationkey', target_column='n_nationkey', role='customer_hierarchy', description='Each customer is associated with a nation.')
JoinEdge(source='nation', target='region', source_column='n_regionkey', target_column='r_regionkey', role='customer_hierarchy', description='Each nation is associated with a region.')


lineitem.l_orderkey → orders.o_orderkey
orders.o_custkey → customer.c_custkey
customer.c_nationkey → nation.n_nationkey
nation.n_regionkey → region.r_regionkey


## Test- SQL Generator

In [23]:
from src.semantic.sql_generator import SQLGenerator

In [24]:
sqlgen = SQLGenerator()

=============== EXAMPLE 1 =================

In [25]:
question = "revenue by region in year 1993"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)
  TABLE PAIR: frozenset({'customer', 'orders'})
    - orders.o_custkey = customer.c_custkey (role=customer_hierarchy)
  TABLE PAIR: frozenset({'customer', 'nation'})
    - customer.c_nationkey = nation.n_nationkey (role=customer_hierarchy)
  TABLE PAIR: frozenset({'region', 'nation'})
    - nation.n_regionkey = region.r_regionkey (role=customer_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: customer
    Other table:   orders
    ON: orders.o_custkey = customer.c_custkey
       edge: orders.o_custkey = customer.c_custkey (role=customer_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: nation
    Other table:   customer
    ON: customer.c_nationkey = natio

=============== EXAMPLE 2 =================

In [26]:
question = "total revenue last month"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)

SELECT
  SUM(l_extendedprice * (1 - l_discount)) AS revenue,
  orders.o_orderdate AS order_date
FROM lineitem
JOIN orders ON lineitem.l_orderkey = orders.o_orderkey
WHERE orders.o_orderdate >= DATE_TRUNC('month', CURRENT_DATE - INTERVAL '1' MONTH) AND orders.o_orderdate < DATE_TRUNC('month', CURRENT_DATE)
GROUP BY orders.o_orderdate


=============== EXAMPLE 3 =================

In [27]:
question = "number of orders this year"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===

SELECT
  COUNT(DISTINCT o_orderkey) AS order_count,
  orders.o_orderdate AS order_date
FROM orders
WHERE orders.o_orderdate >= DATE_TRUNC('year', CURRENT_DATE)
GROUP BY orders.o_orderdate


=============== EXAMPLE 4 =================

In [28]:
question = "revenue by nation this year"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)
  TABLE PAIR: frozenset({'customer', 'orders'})
    - orders.o_custkey = customer.c_custkey (role=customer_hierarchy)
  TABLE PAIR: frozenset({'customer', 'nation'})
    - customer.c_nationkey = nation.n_nationkey (role=customer_hierarchy)
  TABLE PAIR: frozenset({'region', 'nation'})
    - nation.n_regionkey = region.r_regionkey (role=customer_hierarchy)
    - region.r_regionkey = nation.n_regionkey (role=customer_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: customer
    Other table:   orders
    ON: orders.o_custkey = customer.c_custkey
       edge: orders.o_custkey = customer.c_custkey (role=customer_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table:

=============== EXAMPLE 5 =================

In [29]:
question = "revenue by supplier last quarter"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)
  TABLE PAIR: frozenset({'lineitem', 'partsupp'})
    - lineitem.l_suppkey = partsupp.ps_suppkey (role=supplier_hierarchy)
  TABLE PAIR: frozenset({'supplier', 'partsupp'})
    - partsupp.ps_suppkey = supplier.s_suppkey (role=supplier_hierarchy)
  TABLE PAIR: frozenset({'supplier', 'nation'})
    - supplier.s_nationkey = nation.n_nationkey (role=supplier_hierarchy)
    - nation.n_nationkey = supplier.s_nationkey (role=supplier_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: partsupp
    Other table:   lineitem
    ON: lineitem.l_suppkey = partsupp.ps_suppkey
       edge: lineitem.l_suppkey = partsupp.ps_suppkey (role=supplier_hierarchy)


--> DEBUG JOIN BUILD

=============== EXAMPLE 6 =================

In [30]:
question = "quantity sold by product name last 90 days"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)
  TABLE PAIR: frozenset({'lineitem', 'part'})
    - lineitem.l_partkey = part.p_partkey (role=product_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: part
    Other table:   lineitem
    ON: lineitem.l_partkey = part.p_partkey
       edge: lineitem.l_partkey = part.p_partkey (role=product_hierarchy)

SELECT
  SUM(l_quantity) AS quantity_sold,
  orders.o_orderdate AS order_date,
  part.p_name AS part_name
FROM lineitem
JOIN orders ON lineitem.l_orderkey = orders.o_orderkey
JOIN part ON lineitem.l_partkey = part.p_partkey
WHERE orders.o_orderdate >= CURRENT_DATE - INTERVAL '90 days'
GROUP BY orders.o_orderdate, part.p_name


=============== EXAMPLE 7 =================

In [31]:
question = "revenue by supplier this month"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)
  TABLE PAIR: frozenset({'lineitem', 'partsupp'})
    - lineitem.l_suppkey = partsupp.ps_suppkey (role=supplier_hierarchy)
  TABLE PAIR: frozenset({'supplier', 'partsupp'})
    - partsupp.ps_suppkey = supplier.s_suppkey (role=supplier_hierarchy)
  TABLE PAIR: frozenset({'supplier', 'nation'})
    - supplier.s_nationkey = nation.n_nationkey (role=supplier_hierarchy)
    - nation.n_nationkey = supplier.s_nationkey (role=supplier_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: partsupp
    Other table:   lineitem
    ON: lineitem.l_suppkey = partsupp.ps_suppkey
       edge: lineitem.l_suppkey = partsupp.ps_suppkey (role=supplier_hierarchy)


--> DEBUG JOIN BUILD

=============== EXAMPLE 8 =================

In [32]:
question = "revenue by brand last year"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)
  TABLE PAIR: frozenset({'lineitem', 'part'})
    - lineitem.l_partkey = part.p_partkey (role=product_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: part
    Other table:   lineitem
    ON: lineitem.l_partkey = part.p_partkey
       edge: lineitem.l_partkey = part.p_partkey (role=product_hierarchy)

SELECT
  SUM(l_extendedprice * (1 - l_discount)) AS revenue,
  orders.o_orderdate AS order_date,
  part.p_brand AS product_brand
FROM lineitem
JOIN orders ON lineitem.l_orderkey = orders.o_orderkey
JOIN part ON lineitem.l_partkey = part.p_partkey
WHERE orders.o_orderdate >= DATE_TRUNC('year', CURRENT_DATE) - INTERVAL '1 year' AND orders.o_orderdate < DATE_TRUNC('

=============== EXAMPLE 9 =================

In [33]:
question = "average order size by month this year"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)

SELECT
  SUM(l_quantity) / COUNT(DISTINCT l_orderkey) AS average_order_size,
  DATE_TRUNC('month', orders.o_orderdate) AS month_date
FROM lineitem
JOIN orders ON lineitem.l_orderkey = orders.o_orderkey
WHERE orders.o_orderdate >= DATE_TRUNC('year', CURRENT_DATE)
GROUP BY DATE_TRUNC('month', orders.o_orderdate)


=============== EXAMPLE 10 =================

In [34]:
question = "revenue by month this year"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)

SELECT
  SUM(l_extendedprice * (1 - l_discount)) AS revenue,
  DATE_TRUNC('month', orders.o_orderdate) AS month_date
FROM lineitem
JOIN orders ON lineitem.l_orderkey = orders.o_orderkey
WHERE orders.o_orderdate >= DATE_TRUNC('year', CURRENT_DATE)
GROUP BY DATE_TRUNC('month', orders.o_orderdate)


In [35]:
question = "revenue by month last year"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)

SELECT
  SUM(l_extendedprice * (1 - l_discount)) AS revenue,
  DATE_TRUNC('month', orders.o_orderdate) AS month_date
FROM lineitem
JOIN orders ON lineitem.l_orderkey = orders.o_orderkey
WHERE orders.o_orderdate >= DATE_TRUNC('year', CURRENT_DATE) - INTERVAL '1 year' AND orders.o_orderdate < DATE_TRUNC('year', CURRENT_DATE)
GROUP BY DATE_TRUNC('month', orders.o_orderdate)


In [36]:
question = "revenue by month over rolling 12 months"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)

SELECT
  SUM(l_extendedprice * (1 - l_discount)) AS revenue,
  DATE_TRUNC('month', orders.o_orderdate) AS month_date
FROM lineitem
JOIN orders ON lineitem.l_orderkey = orders.o_orderkey
WHERE orders.o_orderdate >= DATEADD(month, -12, CURRENT_DATE)
GROUP BY DATE_TRUNC('month', orders.o_orderdate)


In [37]:
question = "profit by quarter in 1992"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)
  TABLE PAIR: frozenset({'lineitem', 'partsupp'})
    - lineitem.l_partkey = partsupp.ps_partkey (role=product_hierarchy)
    - lineitem.l_suppkey = partsupp.ps_suppkey (role=supplier_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: partsupp
    Other table:   lineitem
    ON: lineitem.l_partkey = partsupp.ps_partkey AND lineitem.l_suppkey = partsupp.ps_suppkey
       edge: lineitem.l_partkey = partsupp.ps_partkey (role=product_hierarchy)
       edge: lineitem.l_suppkey = partsupp.ps_suppkey (role=supplier_hierarchy)

SELECT
  SUM(l_extendedprice * (1 - l_discount) - ps_supplycost * l_quantity) AS profit,
  DATE_TRUNC('quarter', orders.o_orderdate) AS quarter_

In [38]:
question = "profit by quarter in 1992"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(json.dumps(asdict(plan), indent=2))


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)
  TABLE PAIR: frozenset({'lineitem', 'partsupp'})
    - lineitem.l_partkey = partsupp.ps_partkey (role=product_hierarchy)
    - lineitem.l_suppkey = partsupp.ps_suppkey (role=supplier_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: partsupp
    Other table:   lineitem
    ON: lineitem.l_partkey = partsupp.ps_partkey AND lineitem.l_suppkey = partsupp.ps_suppkey
       edge: lineitem.l_partkey = partsupp.ps_partkey (role=product_hierarchy)
       edge: lineitem.l_suppkey = partsupp.ps_suppkey (role=supplier_hierarchy)

{
  "measure": {
    "name": "profit",
    "expression": "SUM(l_extendedprice * (1 - l_discount) - ps_supplycost * l_quantity)",
    "table": "l

In [39]:
question = "profit by quarter in 1992"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print("GENERATED SQL:")
print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)
  TABLE PAIR: frozenset({'lineitem', 'partsupp'})
    - lineitem.l_partkey = partsupp.ps_partkey (role=product_hierarchy)
    - lineitem.l_suppkey = partsupp.ps_suppkey (role=supplier_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: partsupp
    Other table:   lineitem
    ON: lineitem.l_partkey = partsupp.ps_partkey AND lineitem.l_suppkey = partsupp.ps_suppkey
       edge: lineitem.l_partkey = partsupp.ps_partkey (role=product_hierarchy)
       edge: lineitem.l_suppkey = partsupp.ps_suppkey (role=supplier_hierarchy)

GENERATED SQL:
SELECT
  SUM(l_extendedprice * (1 - l_discount) - ps_supplycost * l_quantity) AS profit,
  DATE_TRUNC('quarter', orders.o_orderda

=============== EXAMPLE 10 =================

In [40]:
question = "revenue by supplier and part last 90 days"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)
  TABLE PAIR: frozenset({'lineitem', 'partsupp'})
    - lineitem.l_suppkey = partsupp.ps_suppkey (role=supplier_hierarchy)
  TABLE PAIR: frozenset({'supplier', 'partsupp'})
    - partsupp.ps_suppkey = supplier.s_suppkey (role=supplier_hierarchy)
  TABLE PAIR: frozenset({'supplier', 'nation'})
    - supplier.s_nationkey = nation.n_nationkey (role=supplier_hierarchy)
    - nation.n_nationkey = supplier.s_nationkey (role=supplier_hierarchy)
  TABLE PAIR: frozenset({'lineitem', 'part'})
    - lineitem.l_partkey = part.p_partkey (role=product_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: partsupp
    Other table:   lineitem
    ON: lineitem.l_suppkey = partsupp.

=============== EXAMPLE 11 =================

In [41]:
question = "average order value by customer"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)
  TABLE PAIR: frozenset({'customer', 'orders'})
    - orders.o_custkey = customer.c_custkey (role=customer_hierarchy)
  TABLE PAIR: frozenset({'customer', 'nation'})
    - customer.c_nationkey = nation.n_nationkey (role=customer_hierarchy)
    - nation.n_nationkey = customer.c_nationkey (role=customer_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: customer
    Other table:   orders
    ON: orders.o_custkey = customer.c_custkey
       edge: orders.o_custkey = customer.c_custkey (role=customer_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: nation
    Other table:   customer
    ON: customer.c_nationkey = nation.n_nationkey AND nation.n_nationkey = custo

=============== EXAMPLE 12 =================

In [42]:
question = "revenue by customer and month"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)
  TABLE PAIR: frozenset({'customer', 'orders'})
    - orders.o_custkey = customer.c_custkey (role=customer_hierarchy)
  TABLE PAIR: frozenset({'customer', 'nation'})
    - customer.c_nationkey = nation.n_nationkey (role=customer_hierarchy)
    - nation.n_nationkey = customer.c_nationkey (role=customer_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: customer
    Other table:   orders
    ON: orders.o_custkey = customer.c_custkey
       edge: orders.o_custkey = customer.c_custkey (role=customer_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: nation
    Other table:   customer
    ON: customer.c_nationkey = nation.n_nationkey AND nation.n_nationkey = custo

=============== EXAMPLE 13 =================

In [43]:
question = "quantity sold by supplier and part last 60 days"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)
  TABLE PAIR: frozenset({'lineitem', 'partsupp'})
    - lineitem.l_suppkey = partsupp.ps_suppkey (role=supplier_hierarchy)
  TABLE PAIR: frozenset({'supplier', 'partsupp'})
    - partsupp.ps_suppkey = supplier.s_suppkey (role=supplier_hierarchy)
  TABLE PAIR: frozenset({'supplier', 'nation'})
    - supplier.s_nationkey = nation.n_nationkey (role=supplier_hierarchy)
    - nation.n_nationkey = supplier.s_nationkey (role=supplier_hierarchy)
  TABLE PAIR: frozenset({'lineitem', 'part'})
    - lineitem.l_partkey = part.p_partkey (role=product_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: partsupp
    Other table:   lineitem
    ON: lineitem.l_suppkey = partsupp.

=============== EXAMPLE 14 =================

In [44]:
question = "revenue by nation and supplier last year"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)
  TABLE PAIR: frozenset({'customer', 'orders'})
    - orders.o_custkey = customer.c_custkey (role=customer_hierarchy)
  TABLE PAIR: frozenset({'customer', 'nation'})
    - customer.c_nationkey = nation.n_nationkey (role=customer_hierarchy)
  TABLE PAIR: frozenset({'region', 'nation'})
    - nation.n_regionkey = region.r_regionkey (role=customer_hierarchy)
    - region.r_regionkey = nation.n_regionkey (role=customer_hierarchy)
  TABLE PAIR: frozenset({'lineitem', 'partsupp'})
    - lineitem.l_suppkey = partsupp.ps_suppkey (role=supplier_hierarchy)
  TABLE PAIR: frozenset({'supplier', 'partsupp'})
    - partsupp.ps_suppkey = supplier.s_suppkey (role=supplier_hierarchy)
  TABLE PAIR: frozenset({'supplier', 'nation'})
    - supplier.s_nationkey = nation.n_nationkey (role=supplier_hierarchy)
    - nation.n_nationkey = supplier.s_nationkey (role=supplier_hier

=============== EXAMPLE 15 =================

In [45]:
question = "top 5 regions by revenue last year"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)
  TABLE PAIR: frozenset({'customer', 'orders'})
    - orders.o_custkey = customer.c_custkey (role=customer_hierarchy)
  TABLE PAIR: frozenset({'customer', 'nation'})
    - customer.c_nationkey = nation.n_nationkey (role=customer_hierarchy)
  TABLE PAIR: frozenset({'region', 'nation'})
    - nation.n_regionkey = region.r_regionkey (role=customer_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: customer
    Other table:   orders
    ON: orders.o_custkey = customer.c_custkey
       edge: orders.o_custkey = customer.c_custkey (role=customer_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: nation
    Other table:   customer
    ON: customer.c_nationkey = natio

=============== EXAMPLE 16 =================

In [46]:
question = "revenue by customer segment this quarter"
plan = resolver.plan_from_question(question)
joins = joiner.joins_for_plan(plan)
sql = sqlgen.sql_from_plan(plan, joins)

print(sql)


=== DEBUG: JOIN GROUPS ===
  TABLE PAIR: frozenset({'lineitem', 'orders'})
    - lineitem.l_orderkey = orders.o_orderkey (role=order)
  TABLE PAIR: frozenset({'customer', 'orders'})
    - orders.o_custkey = customer.c_custkey (role=customer_hierarchy)
  TABLE PAIR: frozenset({'customer', 'nation'})
    - customer.c_nationkey = nation.n_nationkey (role=customer_hierarchy)
    - nation.n_nationkey = customer.c_nationkey (role=customer_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: orders
    Other table:   lineitem
    ON: lineitem.l_orderkey = orders.o_orderkey
       edge: lineitem.l_orderkey = orders.o_orderkey (role=order)


--> DEBUG JOIN BUILD:
    Joining table: customer
    Other table:   orders
    ON: orders.o_custkey = customer.c_custkey
       edge: orders.o_custkey = customer.c_custkey (role=customer_hierarchy)


--> DEBUG JOIN BUILD:
    Joining table: nation
    Other table:   customer
    ON: customer.c_nationkey = nation.n_nationkey AND nation.n_nationkey = custo

## Test Embedding

In [47]:
import json
from pathlib import Path
from dataclasses import asdict
from pprint import pprint

from src.semantic.semantic_model import SemanticModel
from src.semantic.semantic_resolver import SemanticResolver
from src.semantic.embedding_store import EmbeddingIndex
from langchain_openai import OpenAIEmbeddings

model = SemanticModel.from_yaml(
    os.path.join(project_root, "src/semantic/model_tpch.yml")
)

# Load embedding index
emb_path = os.path.join(project_root, "artifacts/semantic_embeddings_tpch.json")
with Path(emb_path).open("r", encoding="utf-8") as f:
    emb_payload = json.load(f)

embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")
embedding_index = EmbeddingIndex.from_dict(emb_payload)

resolver = SemanticResolver(model, embedding_index=embedding_index, embedding_model=embedding_model)

### Inspect Embedding Index

In [48]:
resolver.embedding_model

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x00000203658CB980>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000002036590C0E0>, model='text-embedding-3-large', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [49]:
print("\n--- ALL ITEMS IN EMBEDDING INDEX ---")
for it in embedding_index.items:
    print(it.type, "|", it.name)


--- ALL ITEMS IN EMBEDDING INDEX ---
measure | revenue
measure | gross_sales
measure | order_count
measure | avg_discount
measure | quantity_sold
measure | average_order_size
measure | average_quantity
measure | average_revenue
measure | profit
measure | profit_margin_pct
measure | discount_amount
measure | average_order_value
measure | customer_count
measure | supplier_count
measure | arpc
measure | line_count
measure | average_lines_per_order
dimension | order_date
dimension | ship_date
dimension | customer_name
dimension | customer_segment
dimension | region_name
dimension | nation_name
dimension | supplier_name
dimension | part_name
dimension | product_brand


In [50]:
dims = [it for it in embedding_index.items if it.type=="dimension"]
print(len(dims), "dimension embeddings found")
for d in dims:
    print(d.name, "|", d.table)

9 dimension embeddings found
order_date | orders
ship_date | lineitem
customer_name | customer
customer_segment | customer
region_name | region
nation_name | nation
supplier_name | supplier
part_name | part
product_brand | part


In [51]:
vec = resolver._embed_query_text("cust")
print("Vector:", type(vec), vec.shape)

Vector: <class 'numpy.ndarray'> (3072,)


In [52]:
print("=== TOTAL ITEMS IN EMBEDDING INDEX ===")
print(len(embedding_index.items))

dims = [it for it in embedding_index.items if it.type == "dimension"]
print("\n=== TOTAL DIMENSIONS ===")
print(len(dims))

print("\n=== FIRST 10 DIMENSIONS ===")
for d in dims[:10]:
    print(f"- {d.name:20s} | table={d.table:10s} | column={d.column}")

=== TOTAL ITEMS IN EMBEDDING INDEX ===
26

=== TOTAL DIMENSIONS ===
9

=== FIRST 10 DIMENSIONS ===
- order_date           | table=orders     | column=o_orderdate
- ship_date            | table=lineitem   | column=l_shipdate
- customer_name        | table=customer   | column=c_name
- customer_segment     | table=customer   | column=c_mktsegment
- region_name          | table=region     | column=r_name
- nation_name          | table=nation     | column=n_name
- supplier_name        | table=supplier   | column=s_name
- part_name            | table=part       | column=p_name
- product_brand        | table=part       | column=p_brand


In [53]:
vec = resolver._embed_query_text("cust")

print("=== TOP MATCHES FOR 'cust' ===")
results = embedding_index.search_dimension(vec, top_k=10)

for item, score in results:
    print(f"{item.name:20s}  score={score:.4f}")

=== TOP MATCHES FOR 'cust' ===
customer_name         score=0.2858
customer_segment      score=0.2219
order_date            score=0.1721
part_name             score=0.1611
product_brand         score=0.1411
nation_name           score=0.1378
supplier_name         score=0.1348
ship_date             score=0.1257
region_name           score=0.1160


### Helper

In [54]:
def test_query(q):
    print("\n====================================================")
    print("QUESTION:", q)
    print("====================================================")
    
    plan = resolver.plan_from_question(q)
    pprint(asdict(plan))
    return plan

### Tests

#### A. Direct Dimension Retrieval

In [55]:
print("\n--- Test A: Embedding dimension lookup ---")

test_dimension_queries = [
    "cust",                # → customer
    "custmr",              # fuzzy
    "client",              # fuzzy → customer
    "buyer",               # fuzzy → customer
    "nation",              # direct
    "natn",                # fuzzy → nation
    "regn",                # fuzzy → region
    "area",                # fuzzy → region
]

for q in test_dimension_queries:
    print(f"\nDimension for phrase: '{q}'")
    dim = resolver._semantic_best_dimension(q)
    if dim:
        print("→", dim.name, "| table:", dim.table)
    else:
        print("→ NOT FOUND")


--- Test A: Embedding dimension lookup ---

Dimension for phrase: 'cust'
→ customer_name | table: customer

Dimension for phrase: 'custmr'
→ customer_name | table: customer

Dimension for phrase: 'client'
→ customer_name | table: customer

Dimension for phrase: 'buyer'
→ customer_name | table: customer

Dimension for phrase: 'nation'
→ nation_name | table: nation

Dimension for phrase: 'natn'
→ nation_name | table: nation

Dimension for phrase: 'regn'
→ region_name | table: region

Dimension for phrase: 'area'
→ region_name | table: region
